# Banka Müşteri Churn Tahmini

Bu notebook, bir bankanın müşteri verisi üzerinde churn tahmini yapmak amacıyla hazırlanmıştır. Keşifsel analiz, özellik mühendisliği ve birden fazla sınıflandırma algoritmasının karşılaştırılması adım adım ele alınmıştır.

## 1. Kütüphaneler ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

churn = pd.read_csv('Churn.csv')
df = churn.copy()

## 2. Keşifsel Veri Analizi

In [ ]:
df.describe().T

In [ ]:
df.info()

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number])
print('Sayısal sütunlar:')
print(numeric_columns)

### Yardımcı Fonksiyonlar

In [ ]:
def plot_histogram(df, column_name):
    plt.figure(figsize=(5, 3))
    sns.histplot(df[column_name], kde=True)
    plt.title(f'Distribution of {column_name}')
    col_mean = df[column_name].mean()
    col_median = df[column_name].median()
    plt.axvline(col_mean, color='red', linestyle='--', label='Mean')
    plt.axvline(col_median, color='green', linestyle='-', label='Median')
    plt.legend()
    plt.show()

def plot_boxplot(df, column_name):
    plt.figure(figsize=(5, 3))
    sns.boxplot(y=df[column_name])
    plt.title(f'Box Plot of {column_name}')
    plt.ylabel(column_name)
    plt.show()

### 2.1 Korelasyon Matrisi

`Balance` ile `NumOfProducts` arasında yaklaşık -0.3 düzeyinde negatif korelasyon gözlemlenmiştir. 0.5 eşiğinin altında kaldığından bu ilişki ihmal edilmiştir.

In [ ]:
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
cor = numeric_df.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.title('Korelasyon Matrisi')
plt.show()

### 2.2 Hedef Değişken: Exited

Veri setinde sınıf dağılımı %20 churn / %80 kalmış şeklindedir. Bu dengesizlik ilerleyen aşamada SMOTE ile giderilecektir.

In [ ]:
df['Exited'].value_counts()

In [ ]:
%matplotlib inline
print(f'Churn oranı: {2037/9999:.2%}')
sns.countplot(x='Exited', data=df, label='Count', palette='viridis')
plt.title('Churn Dağılımı (0: Kaldı, 1: Ayrıldı)')
plt.show()

### 2.3 Değişken Analizleri

#### CreditScore

Kredi skoru 405'in altında olan tüm müşterilerin churn olduğu görülmüştür (20 gözlem). Bu gözlem doğrultusunda `smallerthan405` ikili değişkeni türetilmiştir.

In [ ]:
df['CreditScore'].describe().T

In [ ]:
sns.boxplot(df['CreditScore'], orient='h')
plt.title('CreditScore Boxplot')
plt.show()

In [ ]:
# Kredi skoru 400'ün altında olanların tamamı churn olmuş
df[df['CreditScore'] < 405].apply({'Exited': 'value_counts'})

In [ ]:
plot_histogram(df, 'CreditScore')

#### Age

Yaş dağılımında üst değerler outlier izlenimi verse de gerçek müşteri verisi olduğundan veri setinde bırakılmış, gruplandırma yoluyla dönüştürülmüştür.

In [ ]:
df['Age'].describe()

In [ ]:
sns.boxplot(df['Age'], orient='h')
plt.title('Age Boxplot')
plt.show()

In [ ]:
plot_histogram(df, 'Age')

#### Tenure

0 ve 10 yıllık müşteri sayısı diğer gruplara kıyasla daha azdır; dağılım genel olarak düzgündür.

In [ ]:
df['Tenure'].value_counts()

In [ ]:
plot_histogram(df, 'Tenure')
plot_boxplot(df, 'Tenure')

#### Balance

Bakiyesi sıfır olan müşterilerin churn oranı, bakiyesi sıfır olmayanlara kıyasla daha düşük çıkmıştır. Bu gözlem `Balance0` ikili değişkeninin türetilmesine zemin hazırlamıştır.

In [ ]:
df['Balance'].describe()

In [ ]:
sns.boxplot(df['Balance'], orient='h')
plt.show()

In [ ]:
print('Balance > 100.000 olanlar:', df[df['Balance'] > 100000]['Exited'].mean())
print('Balance = 0 olanlar:', df[df['Balance'] == 0]['Exited'].mean())
print('Balance != 0 olanlar:', df[df['Balance'] != 0]['Exited'].mean())

In [ ]:
plot_histogram(df, 'Balance')

#### NumOfProducts

3 veya 4 ürüne sahip müşterilerin churn oranı son derece yüksektir (sırasıyla %82 ve %100). Bu bilgi `NOP*` değişkeninin oluşturulmasında kullanılmıştır.

In [ ]:
df['NumOfProducts'].value_counts()

In [ ]:
for n in [1, 2, 3, 4]:
    rate = df[df['NumOfProducts'] == n]['Exited'].mean()
    print(f'NumOfProducts={n}: Churn oranı = {rate:.2%}')

#### HasCrCard, Geography, Gender

Kredi kartı sahipliğinin churn üzerinde anlamlı bir etkisi bulunmamıştır. Alman müşterilerin terk oranı (%32) Fransız (%16) ve İspanyol (%17) müşterilere kıyasla belirgin biçimde yüksektir.

In [ ]:
print('HasCrCard - Exited korelasyonu:', df['HasCrCard'].corr(df['Exited']))
sns.countplot(x='HasCrCard', data=df, palette='viridis')
plt.title('Kredi Kartı Sahipliği')
plt.show()

In [ ]:
gender_counts = df['Gender'].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=['lightcoral', 'lightblue'])
plt.title('Cinsiyet Dağılımı')
plt.show()

## 3. Ön İşleme

### 3.1 Encoding

`Geography` ve `Gender` değişkenlerine one-hot encoding uygulanmıştır.

In [ ]:
print(df.columns)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']] = df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']].astype(int)
print(df.head(2))

### 3.2 Ülkelere Göre Churn Oranları

In [ ]:
alm_total = df[df['Geography_Germany']==1].shape[0]
alm_churn = df[(df['Geography_Germany']==1) & (df['Exited']==1)].shape[0]
print(f'Almanya churn oranı: {alm_churn/alm_total:.2%}')

isp_total = df[df['Geography_Spain']==1].shape[0]
isp_churn = df[(df['Geography_Spain']==1) & (df['Exited']==1)].shape[0]
print(f'İspanya churn oranı: {isp_churn/isp_total:.2%}')

fr = df[(df['Geography_Spain']==0) & (df['Geography_Germany']==0)]
print(f'Fransa churn oranı: {fr["Exited"].mean():.2%}')

### 3.3 Türetilmiş Değişkenler

Veriden çeşitli oranlar ve gruplandırmalar türetilerek modele ek bilgi sağlanmıştır.

In [ ]:
df1 = df.copy()

# Age gruplandırma: 18-30=1, 30-40=2, 40-50=3, 50-60=4, 60-92=5
df1.loc[(df1['Age']>=18) & (df1['Age']<=30), 'Age'] = 1
df1.loc[(df1['Age']>30) & (df1['Age']<=40), 'Age'] = 2
df1.loc[(df1['Age']>40) & (df1['Age']<=50), 'Age'] = 3
df1.loc[(df1['Age']>50) & (df1['Age']<=60), 'Age'] = 4
df1.loc[(df1['Age']>60) & (df1['Age']<=92), 'Age'] = 5

In [ ]:
# Kredi skoru gruplama (uluslararası tabloya göre)
def kredi_skor_tablosu(row):
    k = row.CreditScore
    if k < 300: return 1
    elif k < 500: return 2
    elif k < 601: return 3
    elif k < 661: return 4
    elif k < 781: return 5
    elif k < 851: return 6
    else: return 7

df1 = df1.assign(credit_score_table=df1.apply(lambda x: kredi_skor_tablosu(x), axis=1))

In [ ]:
# Emeklilik değişkeni (Almanya & İspanya: 65, Fransa: 66)
df1['retired'] = df['Age']
df1.loc[(df1['retired']>=65) & (df1['Geography_Germany']==1), 'retired'] = 1
df1.loc[(df1['retired']>=65) & (df1['Geography_Spain']==1), 'retired'] = 1
df1.loc[(df1['retired']>=66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 1
df1.loc[(df1['retired']<65) & (df1['Geography_Germany']==1), 'retired'] = 0
df1.loc[(df1['retired']<65) & (df1['Geography_Spain']==1), 'retired'] = 0
df1.loc[(df1['retired']<66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 0

# Tenure/NumOfProducts
df1['Tenure/NumOfProducts'] = df1['Tenure'] / df1['NumOfProducts']

In [ ]:
# CreditScore < 405 ikili değişken
df1['smallerthan405'] = df['CreditScore']
df1.loc[df1['smallerthan405'] < 405, 'smallerthan405'] = 1
df1.loc[df1['smallerthan405'] > 405, 'smallerthan405'] = 0

# NOP*: NumOfProducts churn riskine göre sıralanmış
# NOP=1: %27 churn | NOP=2: %7 churn | NOP=3: %82 | NOP=4: %100
df1['NOP*'] = df['NumOfProducts']
df1.loc[df1['NOP*'] == 2, 'NOP*'] = 1
df1.loc[df1['NOP*'] == 1, 'NOP*'] = 2
df1.loc[df1['NOP*'] > 2, 'NOP*'] = 3

# Balance0: Balance sıfır mı değil mi?
df1['Balance0'] = df1['Balance']
df1.loc[df1['Balance0'] == 0, 'Balance0'] = 0
df1.loc[df1['Balance0'] != 0, 'Balance0'] = 1

In [ ]:
# Maaş bazlı türetilmiş değişkenler
df1['ES/Age'] = df1['EstimatedSalary'] / (df['Age'] - 17)
df1['Tenure/Age'] = df1['Tenure'] / (df['Age'] - 17)
df1['Balance/ES'] = df1['Balance'] / df1['EstimatedSalary']

# Aylık tahmini maaş
df1['EstimatedSalary'] = df1['EstimatedSalary'] / 12
df1['ES/Tenure'] = df1['EstimatedSalary'] / (df1['Tenure'] + 1)  # +1: sıfıra bölmeyi önler
df1['ES/Score'] = df1['EstimatedSalary'] / df1['credit_score_table']

In [ ]:
# Orijinal değişkenleri çıkar (türetilmişlerle yer değiştirdi)
df1 = df1.drop(['CreditScore', 'Tenure', 'Balance'], axis=1)
df1.head(3)

### 3.4 Ölçekleme

Sayısal değişkenlere Robust Scaler uygulanmıştır. Standart Scaler yerine bu yöntemin tercih edilmesinin nedeni aykırı değerlere karşı daha dayanıklı olmasıdır.

In [ ]:
from sklearn.preprocessing import RobustScaler

df1_num = df1[['Age', 'NumOfProducts', 'EstimatedSalary',
               'credit_score_table', 'Tenure/NumOfProducts', 'NOP*',
               'ES/Age', 'Tenure/Age', 'Balance/ES', 'ES/Tenure', 'ES/Score']]

col = df1_num.columns
x_transformed = pd.DataFrame(RobustScaler().fit(df1_num).transform(df1_num), columns=col)
x_transformed.head()

In [ ]:
# X: Bağımsız değişkenler birleştirme
X = pd.concat([
    x_transformed.loc[:, 'Age':'NumOfProducts'],
    df1.loc[:, 'HasCrCard':'IsActiveMember'],
    x_transformed.loc[:, 'EstimatedSalary'],
    df1.loc[:, 'Geography_Germany':'Gender_Male'],
    x_transformed.loc[:, 'credit_score_table'],
    df1.loc[:, 'retired'],
    x_transformed.loc[:, 'Tenure/NumOfProducts'],
    df1.loc[:, 'smallerthan405'],
    x_transformed.loc[:, 'NOP*'],
    df1.loc[:, 'Balance0'],
    x_transformed.loc[:, 'ES/Age':'ES/Score']
], axis=1)
X.head(2)

## 4. Modelleme

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter

pd.set_option('display.max_columns', None)

In [ ]:
y = df1['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=12345)

### 4.1 Random Forest

In [ ]:
rf_model = RandomForestClassifier().fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

cv_results = cross_val_score(rf_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
importance = rf_model.feature_importances_
plt.figure(figsize=(10, 10))
plt.barh(X.columns, importance)
plt.title('Feature Importance - Random Forest')
plt.show()

### 4.2 Gradient Boosting

In [ ]:
gbm_model = GradientBoostingClassifier().fit(X_train, y_train)
y_pred = gbm_model.predict(X_test)

cv_results = cross_val_score(gbm_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 4.3 LightGBM

LightGBM için hiperparametre araması da yapılmış ancak varsayılan değerlerin üzerinde bir iyileştirme sağlanamamıştır.

In [ ]:
lgbm_model = LGBMClassifier().fit(X_train, y_train)
y_pred = lgbm_model.predict(X_test)

cv_results = cross_val_score(lgbm_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 5. Sınıf Dengesizliği ve SMOTE

Veri setindeki %20/%80 dağılımı modelin azınlık sınıfını öğrenmesini zorlaştırmaktadır. SMOTE ile sentetik örnekler üretilerek sınıflar dengelenmiş, ardından modeller yeniden eğitilmiştir.

In [ ]:
smt = SMOTE(random_state=12345)
X_res, y_res = smt.fit_resample(X, y)
print('SMOTE öncesi dağılım:', Counter(y))
print('SMOTE sonrası dağılım:', Counter(y_res))

### 5.1 LightGBM + SMOTE

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.20, random_state=12345)

lgbm_model = LGBMClassifier(random_state=12345).fit(X_train, y_train)
y_pred = lgbm_model.predict(X_test)
y_train_pred = lgbm_model.predict(X_train)

cv_train = cross_val_score(lgbm_model, X_train, y_train, cv=10, scoring='accuracy')
cv_test  = cross_val_score(lgbm_model, X_test,  y_test,  cv=10, scoring='accuracy')

print('CV Doğruluk (train):', cv_train.mean())
print('CV Doğruluk (test): ', cv_test.mean())
print('Accuracy (train):   ', accuracy_score(y_train, y_train_pred))
print('Accuracy (test):    ', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 5.2 Hata Matrisi

In [ ]:
cf_matrix = confusion_matrix(y_test, y_pred)
print(cf_matrix)
sns.heatmap(cf_matrix / np.sum(cf_matrix), annot=True,
            fmt='.2%', cmap='Blues')
plt.title('Confusion Matrix - LightGBM + SMOTE')
plt.show()

## 6. Sonuç ve Değerlendirme

SMOTE uygulanmış veri seti üzerinde eğitilen Random Forest ve LightGBM modelleri birbirine yakın performans sergilemiştir.

| Model | CV Doğruluk | Train Doğruluk | Test Doğruluk |
|---|---|---|---|
| Random Forest + SMOTE | %89.24 | %92.61 | %89.39 |
| LightGBM + SMOTE | %89.47 | %92.61 | %89.45 |

En iyi sonuç LightGBM + SMOTE kombinasyonundan elde edilmiştir. 3186 tahmin üzerinden %89.42 doğruluk sağlanmış; churn olan müşteriler için precision %92, recall %87, f1-score %89 olarak gerçekleşmiştir.

Öne çıkan değişkenler `Age` ve `NumOfProducts` olmuştur. Türetilmiş değişkenlerden `Tenure/Age`, `ES/Tenure` ve `Geography_Germany` da modele kayda değer katkı sağlamıştır. `retired` ve `smallerthan405` değişkenleri feature importance açısından düşük kalmış olmakla birlikte veri setinde bırakılmıştır.